# 🧠 Урок з нейронних мереж — Fashion MNIST

Інтерактивний урок для 11 класу. Ми разом побудуємо нейронну мережу,
яка вчиться розпізнавати одяг за фотографією 28×28 пікселів.

**Як запустити:**
1. Виконати всі клітинки по порядку (`Cell → Run All`).
2. Остання клітинка запускає Gradio-додаток із 7 вкладками.

**Дані:** очікуються файли `fashion-mnist_train.csv` і `fashion-mnist_test.csv`
у теці `./data/` (локально) або `/kaggle/input/fashionmnist/` (на Kaggle).
Завантажити: <https://www.kaggle.com/datasets/zalando-research/fashionmnist>


## Встановлення залежностей

In [ ]:
# !pip install torch gradio pandas numpy matplotlib seaborn pillow

## Імпорти

In [ ]:
import io
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import gradio as gr

# Для коректного рендерингу графіків у Gradio.
matplotlib.use("Agg")
plt.rcParams.update({"font.size": 11, "axes.titlesize": 13, "figure.dpi": 100})

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Використовуємо: {DEVICE}")

## Дані: константи та завантаження Fashion MNIST

Кожне зображення — 28×28 пікселів у відтінках сірого, тобто 784 числа.
Класів — 10. У CSV перший стовпчик — мітка, інші 784 — пікселі від 0 до 255.

In [ ]:
# Назви класів українською з емодзі — щоб учням було легше впізнавати.
CLASSES_UA = [
    "👕 Футболка", "👖 Штани", "👔 Светр", "👗 Сукня", "🧥 Пальто",
    "👡 Сандаль", "👘 Сорочка", "👟 Кросівок", "👜 Сумка", "👢 Чобіт",
]
IMG_SIDE = 28
INPUT_SIZE = IMG_SIDE * IMG_SIDE
NUM_CLASSES = 10


def find_dataset_files():
    return "/kaggle/input/datasets/organizations/zalando-research/fashionmnist/fashion-mnist_train.csv", "/kaggle/input/datasets/organizations/zalando-research/fashionmnist/fashion-mnist_test.csv"


class State:
    def __init__(self):
        self.X_train = None
        self.y_train = None
        self.X_test = None
        self.y_test = None
        self.data_loaded = False
        self.model = None
        # Список dict-ів по епохах: точність, втрата, ваги першого шару, передбачення.
        self.history = []
        self.architecture = None
        # Кешуємо вибір тестових індексів, щоб «Показати ще 10» працювало.
        self.last_test_indices = None


state = State()


def load_data():
    """Завантажує датасет і нормалізує піксельні значення в [0, 1]."""
    if state.data_loaded:
        return True, "Дані вже завантажено."

    train_path, test_path = find_dataset_files()
    if train_path is None:
        msg = (
            "❌ Не знайдено файлів fashion-mnist_train.csv та fashion-mnist_test.csv.\n\n"
            "Завантажте їх із Kaggle:\n"
            "https://www.kaggle.com/datasets/zalando-research/fashionmnist\n\n"
            "І покладіть у теку ./data/ поряд із цим ноутбуком."
        )
        return False, msg

    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    state.X_train = train_df.iloc[:, 1:].values.astype("float32") / 255.0
    state.y_train = train_df.iloc[:, 0].values.astype("int64")
    state.X_test = test_df.iloc[:, 1:].values.astype("float32") / 255.0
    state.y_test = test_df.iloc[:, 0].values.astype("int64")
    state.data_loaded = True

    return True, (
        f"✅ Дані завантажено: {len(state.X_train)} тренувальних "
        f"та {len(state.X_test)} тестових зображень."
    )

## Модель PyTorch

Проста повнозв'язна мережа: входи → один або кілька прихованих шарів із ReLU
та (опціонально) Dropout → 10 виходів (по одному на клас).

In [ ]:
class FashionNet(nn.Module):
    """Проста повнозв'язна нейронна мережа з налаштовуваною кількістю шарів."""

    def __init__(self, hidden_layers, neurons_per_layer, dropout):
        super().__init__()
        layers = []
        prev = INPUT_SIZE
        for _ in range(hidden_layers):
            layers.append(nn.Linear(prev, neurons_per_layer))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = neurons_per_layer
        layers.append(nn.Linear(prev, NUM_CLASSES))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

    def first_linear(self):
        """Перший Linear-шар — для візуалізації ваг у табі 5."""
        for layer in self.net:
            if isinstance(layer, nn.Linear):
                return layer
        return None


def count_parameters(hidden_layers, neurons_per_layer):
    """Підраховує загальну кількість параметрів (без створення моделі)."""
    total = 0
    prev = INPUT_SIZE
    for _ in range(hidden_layers):
        total += prev * neurons_per_layer + neurons_per_layer
        prev = neurons_per_layer
    total += prev * NUM_CLASSES + NUM_CLASSES
    return total

## Візуалізації

Допоміжні функції, які малюють схему мережі, картинки одягу, ваги нейронів,
графіки точності/втрати та матрицю помилок.

In [ ]:
def fig_to_pil(fig):
    """Перетворює matplotlib-фігуру на PIL-зображення для gr.Image."""
    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)
    return Image.open(buf)


def draw_network_diagram(layer_sizes, highlight=None):
    """Малює схему мережі прямокутниками-шарами та стрілками між ними.

    highlight — назва концепції для підсвічування ('neuron', 'weights' тощо).
    """
    fig, ax = plt.subplots(figsize=(11, 5))
    n_layers = len(layer_sizes)
    x_positions = np.linspace(0.5, n_layers - 0.5, n_layers)

    layer_titles = (
        ["Вхідний\nшар"]
        + [f"Прихований\nшар {i+1}" for i in range(n_layers - 2)]
        + ["Вихідний\nшар"]
    )

    box_colors = ["#cfe8ff"] * n_layers
    if highlight == "layer":
        box_colors = ["#ffd97a"] * n_layers
    if highlight == "neuron" and n_layers > 2:
        box_colors[1] = "#ffd97a"

    for i, (x, size, title) in enumerate(zip(x_positions, layer_sizes, layer_titles)):
        rect = mpatches.FancyBboxPatch(
            (x - 0.25, 0.2), 0.5, 0.6,
            boxstyle="round,pad=0.02",
            linewidth=2, edgecolor="#1f4e79", facecolor=box_colors[i],
        )
        ax.add_patch(rect)
        ax.text(x, 0.5, f"{size}\nнейронів", ha="center", va="center",
                fontsize=12, fontweight="bold")
        ax.text(x, 0.05, title, ha="center", va="top", fontsize=11)

    arrow_color = "#cc3333" if highlight == "weights" else "#666"
    arrow_lw = 3 if highlight == "weights" else 1.5
    for i in range(n_layers - 1):
        ax.annotate(
            "", xy=(x_positions[i+1] - 0.27, 0.5),
            xytext=(x_positions[i] + 0.27, 0.5),
            arrowprops=dict(arrowstyle="->", color=arrow_color, lw=arrow_lw),
        )
        if highlight == "weights":
            ax.text((x_positions[i] + x_positions[i+1]) / 2, 0.6,
                    "ваги", ha="center", color="#cc3333", fontsize=10, fontweight="bold")
        if highlight == "forward":
            ax.text((x_positions[i] + x_positions[i+1]) / 2, 0.6,
                    "→", ha="center", color="#0a7a0a", fontsize=14, fontweight="bold")
        if highlight == "backprop":
            ax.text((x_positions[i] + x_positions[i+1]) / 2, 0.6,
                    "←", ha="center", color="#a000a0", fontsize=14, fontweight="bold")

    ax.set_xlim(0, n_layers)
    ax.set_ylim(-0.1, 1.0)
    ax.axis("off")
    title_map = {
        "neuron": "Нейрон — базова обчислювальна одиниця",
        "layer": "Шар — група нейронів на одному рівні",
        "weights": "Ваги — числа, які мережа навчається змінювати",
        "forward": "Прямий прохід — дані рухаються від входу до виходу",
        "backprop": "Зворотне поширення — рух помилки назад",
    }
    ax.set_title(title_map.get(highlight, "Архітектура нейронної мережі"))
    return fig_to_pil(fig)


def render_image_grid(images, labels=None, predictions=None, confidences=None, ncols=5):
    """Малює сітку зображень із підписами. Якщо є передбачення — фарбує в зелений/червоний."""
    n = len(images)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 1.8, nrows * 2.2))
    axes = np.array(axes).reshape(nrows, ncols)
    for idx in range(nrows * ncols):
        ax = axes[idx // ncols, idx % ncols]
        if idx < n:
            ax.imshow(images[idx].reshape(IMG_SIDE, IMG_SIDE), cmap="gray")
            if predictions is not None:
                correct = predictions[idx] == labels[idx]
                color = "#0a7a0a" if correct else "#cc0000"
                conf_str = f" ({confidences[idx]*100:.0f}%)" if confidences is not None else ""
                ax.set_title(
                    f"Правильно: {CLASSES_UA[labels[idx]]}\n"
                    f"Мережа: {CLASSES_UA[predictions[idx]]}{conf_str}",
                    fontsize=8, color=color,
                )
            elif labels is not None:
                ax.set_title(CLASSES_UA[labels[idx]], fontsize=9)
        ax.axis("off")
    plt.tight_layout()
    return fig_to_pil(fig)


def render_pixel_matrix(image_flat, label):
    """Показує одне зображення як таблицю чисел 0–255 поряд із самим зображенням."""
    img = (image_flat * 255).astype(int).reshape(IMG_SIDE, IMG_SIDE)
    fig, axes = plt.subplots(1, 2, figsize=(13, 6))

    axes[0].imshow(img, cmap="gray")
    axes[0].set_title(f"Так бачимо ми: {CLASSES_UA[label]}")
    axes[0].axis("off")

    axes[1].imshow(img, cmap="gray", alpha=0.3)
    for i in range(IMG_SIDE):
        for j in range(IMG_SIDE):
            v = img[i, j]
            if v > 0:
                color = "white" if v < 128 else "black"
                axes[1].text(j, i, f"{v}", ha="center", va="center",
                             fontsize=4.5, color=color)
    axes[1].set_title("Так бачить комп'ютер: матриця 28×28 чисел від 0 до 255")
    axes[1].axis("off")
    plt.tight_layout()
    return fig_to_pil(fig)


def render_class_histogram():
    """Гістограма кількості зображень кожного класу в тренувальній вибірці."""
    counts = np.bincount(state.y_train, minlength=NUM_CLASSES)
    fig, ax = plt.subplots(figsize=(10, 4))
    bars = ax.bar(range(NUM_CLASSES), counts, color="#4a90d9")
    ax.set_xticks(range(NUM_CLASSES))
    ax.set_xticklabels(CLASSES_UA, rotation=30, ha="right")
    ax.set_ylabel("Кількість зображень")
    ax.set_title("Скільки зображень кожного класу в тренувальних даних")
    for bar, c in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 50,
                str(c), ha="center", fontsize=9)
    plt.tight_layout()
    return fig


def render_weight_filters(weights, n_show=10):
    """Показує n_show нейронів першого шару як зображення 28×28."""
    fig, axes = plt.subplots(2, 5, figsize=(10, 4.5))
    axes = axes.flatten()
    n_show = min(n_show, weights.shape[0])
    vmax = np.abs(weights[:n_show]).max()
    for i in range(10):
        if i < n_show:
            w = weights[i].reshape(IMG_SIDE, IMG_SIDE)
            axes[i].imshow(w, cmap="bwr", vmin=-vmax, vmax=vmax)
            axes[i].set_title(f"Нейрон {i+1}", fontsize=10)
        axes[i].axis("off")
    fig.suptitle(
        "На що «дивиться» кожен нейрон першого шару\n"
        "(червоне = активує, синє = пригнічує)",
        fontsize=11,
    )
    plt.tight_layout()
    return fig_to_pil(fig)


def render_curve(history, key_train, key_test, title, ylabel):
    """Графік метрики по епохах: train vs test."""
    fig, ax = plt.subplots(figsize=(8, 4.5))
    epochs = [h["epoch"] for h in history]
    ax.plot(epochs, [h[key_train] for h in history], "o-",
            label="Тренувальна вибірка", color="#4a90d9")
    ax.plot(epochs, [h[key_test] for h in history], "s-",
            label="Тестова вибірка", color="#d94a4a")
    ax.set_xlabel("Епоха")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.tight_layout()
    return fig


def render_confusion_matrix(y_true, y_pred):
    """Матриця помилок — які класи мережа плутає."""
    cm = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=CLASSES_UA, yticklabels=CLASSES_UA, ax=ax,
                cbar_kws={"label": "Кількість прикладів"})
    ax.set_xlabel("Що сказала мережа")
    ax.set_ylabel("Правильна відповідь")
    ax.set_title("Матриця помилок — що мережа плутає з чим")
    plt.xticks(rotation=30, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()
    return fig


def per_class_accuracy_table(y_true, y_pred):
    """Точність по кожному класу окремо — як pandas DataFrame для gr.Dataframe."""
    rows = []
    for c in range(NUM_CLASSES):
        mask = y_true == c
        acc = (y_pred[mask] == c).mean() if mask.sum() else 0.0
        rows.append({
            "Клас": CLASSES_UA[c],
            "Точність, %": round(float(acc) * 100, 1),
            "Кількість тестових": int(mask.sum()),
        })
    return pd.DataFrame(rows).sort_values("Точність, %", ascending=False)

## Тренування

`train_model` створює нову мережу за вибраними параметрами, проходить по даних
указану кількість епох і після кожної епохи зберігає метрики, ваги першого шару
та передбачення на тесті — щоб усе було готове для табів 5 і 6.

In [ ]:
def evaluate(model, X, y, batch_size=512):
    """Рахує середню втрату й точність на наборі (X, y), а також передбачення."""
    model.eval()
    criterion = nn.CrossEntropyLoss(reduction="sum")
    loader = DataLoader(
        TensorDataset(torch.from_numpy(X), torch.from_numpy(y)),
        batch_size=batch_size, shuffle=False,
    )
    total_loss = 0.0
    total_correct = 0
    all_preds, all_confs = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits = model(xb)
            total_loss += criterion(logits, yb).item()
            probs = torch.softmax(logits, dim=1)
            confs, preds = probs.max(dim=1)
            total_correct += (preds == yb).sum().item()
            all_preds.append(preds.cpu().numpy())
            all_confs.append(confs.cpu().numpy())
    n = len(X)
    return total_loss / n, total_correct / n, np.concatenate(all_preds), np.concatenate(all_confs)


def train_model(hidden_layers, neurons_per_layer, dropout, learning_rate,
                batch_size, num_epochs, progress=gr.Progress()):
    """Тренує мережу й зберігає історію по епохах. Повертає підсумковий текст."""
    ok, msg = load_data()
    if not ok:
        return msg

    progress(0, desc="Готуємо дані…")

    model = FashionNet(int(hidden_layers), int(neurons_per_layer), float(dropout)).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=float(learning_rate))
    criterion = nn.CrossEntropyLoss()

    train_ds = TensorDataset(torch.from_numpy(state.X_train), torch.from_numpy(state.y_train))
    train_loader = DataLoader(train_ds, batch_size=int(batch_size), shuffle=True)

    history = []
    num_epochs = int(num_epochs)
    for epoch in range(1, num_epochs + 1):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()

        # Оцінюємо на тренувальній і тестовій вибірках наприкінці кожної епохи.
        train_loss, train_acc, _, _ = evaluate(model, state.X_train, state.y_train)
        test_loss, test_acc, test_preds, test_confs = evaluate(model, state.X_test, state.y_test)

        # Зберігаємо ваги першого шару — щоб потім показати «що бачить нейрон».
        first_w = model.first_linear().weight.detach().cpu().numpy().copy()

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "test_loss": test_loss,
            "test_acc": test_acc,
            "first_layer_weights": first_w,
            "test_preds": test_preds,
            "test_confs": test_confs,
        })

        progress(epoch / num_epochs,
                 desc=f"Епоха {epoch}/{num_epochs} — точність {test_acc*100:.1f}%")

    state.model = model
    state.history = history
    state.architecture = {
        "hidden_layers": int(hidden_layers),
        "neurons_per_layer": int(neurons_per_layer),
        "dropout": float(dropout),
        "learning_rate": float(learning_rate),
        "batch_size": int(batch_size),
        "num_epochs": num_epochs,
    }

    final = history[-1]
    return (
        f"✅ Готово! Тренування завершено.\n\n"
        f"Фінальна точність на тестових даних: **{final['test_acc']*100:.2f}%**\n"
        f"Фінальна точність на тренувальних даних: **{final['train_acc']*100:.2f}%**\n\n"
        f"Тепер перейдіть на вкладки «Епохи» та «Порівняння», щоб дослідити навчання."
    )

## Тексти пояснень концепцій

Текстові пояснення для табу 1 — простою мовою, з аналогіями, без формул.

In [ ]:
CONCEPT_EXPLANATIONS = {
    "Нейрон": (
        "**Нейрон** — найпростіша обчислювальна одиниця мережі. "
        "Він отримує числа (входи), множить кожен на свій коефіцієнт (вагу), "
        "додає все разом — і передає результат далі.\n\n"
        "🧠 **Аналогія:** як нейрон у мозку, який «спалахує», коли отримує "
        "достатньо сильний сигнал від сусідів."
    ),
    "Шар": (
        "**Шар** — група нейронів, що обробляють дані на одному рівні.\n\n"
        "- **Вхідний шар** — приймає сирі дані (наші 784 пікселі).\n"
        "- **Прихований шар** — шукає закономірності всередині (наприклад, контури).\n"
        "- **Вихідний шар** — дає відповідь (10 чисел — по одному на кожен клас одягу)."
    ),
    "Ваги": (
        "**Ваги** — числа, які мережа змінює під час навчання. "
        "Це її «пам\u2019ять».\n\n"
        "Спочатку ваги випадкові, тому мережа вгадує навмання. "
        "Поступово вона підкручує їх так, щоб давати правильні відповіді."
    ),
    "Функція активації (ReLU)": (
        "**Функція активації** перетворює суму на виході нейрона.\n\n"
        "**ReLU** дуже проста:\n"
        "- якщо число ≥ 0 → залишається, як є\n"
        "- якщо число < 0 → стає 0\n\n"
        "Завдяки цьому мережа вміє знаходити нелінійні закономірності — "
        "складніші, ніж пряма лінія."
    ),
    "Прямий прохід": (
        "**Прямий прохід (forward pass)** — це коли дані рухаються "
        "крізь мережу зліва направо: від картинки на вході до відповіді на виході.\n\n"
        "На кожному кроці нейрони рахують свій вихід і передають його наступному шару."
    ),
    "Функція втрат": (
        "**Функція втрат** показує, наскільки мережа помилилась.\n\n"
        "- Близько до 0 — мережа вгадує добре\n"
        "- Велике число — мережа сильно помиляється\n\n"
        "Мета навчання — зробити втрату якомога меншою."
    ),
    "Зворотне поширення": (
        "**Зворотне поширення (backpropagation)** — алгоритм, який рахує, "
        "як саме треба змінити кожну вагу, щоб зменшити втрату.\n\n"
        "Він рухається у зворотному напрямку — від виходу до входу — і для кожної "
        "ваги обчислює, наскільки саме вона винна в помилці."
    ),
    "Градієнтний спуск": (
        "**Градієнтний спуск** — крок у напрямку зменшення втрати.\n\n"
        "Уявіть, що ви на горі в тумані й хочете спуститись униз. "
        "Ви робите крок туди, де земля знижується. І так багато разів.\n\n"
        "**Learning rate** — розмір кроку. Малий — повільно, але стабільно. "
        "Великий — швидко, але можна перестрибнути яму."
    ),
    "Епоха": (
        "**Епоха** — один повний прохід мережі по всіх тренувальних зображеннях.\n\n"
        "Зазвичай потрібно багато епох (5–50), щоб мережа вивчила дані добре."
    ),
    "Батч": (
        "**Батч (batch)** — скільки картинок мережа дивиться за один крок, "
        "перш ніж оновити ваги.\n\n"
        "- Малий батч (16) — частіші оновлення, але шумніші\n"
        "- Великий батч (128) — стабільніші оновлення, але рідші"
    ),
    "Перенавчання": (
        "**Перенавчання (overfitting)** — мережа «зазубрила» тренувальні дані "
        "й погано працює на нових.\n\n"
        "📊 Ознака: точність на тренуванні висока, на тесті — низька. "
        "Це наче учень, який вивчив відповіді з підручника, але не розуміє теми."
    ),
    "Dropout": (
        "**Dropout** — техніка проти перенавчання.\n\n"
        "Під час тренування мережа випадково «вимикає» частину нейронів "
        "(наприклад, 20% або 40%). Через це вона не покладається на якогось "
        "одного нейрона й вчиться надійніше."
    ),
}

CONCEPT_TO_HIGHLIGHT = {
    "Нейрон": "neuron",
    "Шар": "layer",
    "Ваги": "weights",
    "Прямий прохід": "forward",
    "Зворотне поширення": "backprop",
}

## Будівельні блоки вкладок

Кожна функція `build_tab_*` додає свою вкладку у Gradio-додаток.

In [ ]:
def build_tab_intro():
    """Таб 1: пояснення базових понять із підсвічуваною схемою."""
    gr.Markdown(
        "### 🧠 Що таке нейронна мережа\n"
        "Сьогодні ми вивчимо, як працює нейронна мережа, і навчимо її розпізнавати одяг. "
        "Виберіть концепцію зліва — побачите її пояснення та підсвічування на схемі."
    )
    with gr.Row():
        with gr.Column(scale=1):
            concept = gr.Radio(
                choices=list(CONCEPT_EXPLANATIONS.keys()),
                value="Нейрон",
                label="Концепції — натисніть, щоб дізнатися більше",
            )
            explanation = gr.Markdown(CONCEPT_EXPLANATIONS["Нейрон"])
        with gr.Column(scale=2):
            diagram = gr.Image(
                value=draw_network_diagram([784, 128, 64, 10], highlight="neuron"),
                label="Схема мережі",
                height=350,
            )

    def update(concept_name):
        text = CONCEPT_EXPLANATIONS[concept_name]
        highlight = CONCEPT_TO_HIGHLIGHT.get(concept_name)
        img = draw_network_diagram([784, 128, 64, 10], highlight=highlight)
        return text, img

    concept.change(update, inputs=concept, outputs=[explanation, diagram])


def build_tab_data():
    """Таб 2: 10 прикладів, одне зображення як матриця чисел, гістограма класів."""
    gr.Markdown(
        "### 📷 Що саме «бачить» наша мережа\n"
        "Подивимось на самі картинки та зрозуміємо, як вони виглядають для комп'ютера."
    )
    sample_grid = gr.Image(label="10 випадкових прикладів із тренувальних даних", height=320)
    pixel_view = gr.Image(label="Одна картинка як числа", height=380)
    gr.Markdown(
        "**Чому ми ділимо на 255?** Кожен піксель — число від 0 (чорний) до 255 (білий). "
        "Якщо подати в мережу великі числа, їй важче навчатись. Тому ми ділимо все на 255 — "
        "тоді кожен піксель стає числом від 0 до 1. Це називається **нормалізацією**."
    )
    histogram = gr.Plot(label="Скільки картинок у кожному класі")
    refresh_btn = gr.Button("🔄 Показати інші зображення", variant="primary")
    status = gr.Markdown()

    def show_examples():
        ok, msg = load_data()
        if not ok:
            return None, None, None, msg
        # Беремо по одному зображенню з кожного класу й перемішуємо.
        idxs = []
        for c in range(NUM_CLASSES):
            class_idxs = np.where(state.y_train == c)[0]
            idxs.append(np.random.choice(class_idxs))
        idxs = np.array(idxs)
        np.random.shuffle(idxs)
        grid_img = render_image_grid(state.X_train[idxs], state.y_train[idxs], ncols=5)
        single_idx = np.random.randint(len(state.X_train))
        pixel_img = render_pixel_matrix(state.X_train[single_idx], state.y_train[single_idx])
        hist_fig = render_class_histogram()
        return grid_img, pixel_img, hist_fig, msg

    refresh_btn.click(show_examples, outputs=[sample_grid, pixel_view, histogram, status])
    return sample_grid, pixel_view, histogram, status, show_examples


def build_tab_architecture():
    """Таб 3: повзунки + жива схема + кількість параметрів."""
    gr.Markdown(
        "### 🏗️ Будуємо нашу мережу\n"
        "Налаштуйте параметри — і відразу побачите, як виглядатиме мережа та скільки в неї «пам'яті» (ваг)."
    )
    with gr.Row():
        with gr.Column():
            hidden_layers = gr.Radio([1, 2, 3], value=2, label="Скільки прихованих шарів")
            neurons = gr.Radio([32, 64, 128, 256, 512], value=128, label="Нейронів у кожному шарі")
            dropout = gr.Radio([0.0, 0.2, 0.4], value=0.2, label="Dropout (захист від перенавчання)")
            learning_rate = gr.Radio([0.01, 0.001, 0.0001], value=0.001, label="Learning rate (швидкість навчання)")
            batch_size = gr.Radio([16, 32, 64, 128], value=64, label="Batch size (картинок за крок)")
            epochs = gr.Slider(1, 20, value=5, step=1, label="Кількість епох")
        with gr.Column():
            arch_diagram = gr.Image(label="Схема вашої мережі", height=320)
            params_info = gr.Markdown()

    def render_architecture(hl, n, dr, lr, bs, ep):
        layer_sizes = [INPUT_SIZE] + [int(n)] * int(hl) + [NUM_CLASSES]
        diagram = draw_network_diagram(layer_sizes)
        n_params = count_parameters(int(hl), int(n))
        text = (
            f"### 📊 Підсумок\n"
            f"- **Загалом параметрів (ваг):** {n_params:,}\n"
            f"- **Архітектура:** {' → '.join(str(x) for x in layer_sizes)}\n\n"
            f"**Що це означає:**\n"
            f"- Більше нейронів ({n}) → більше параметрів → складніші візерунки, але можливе перенавчання\n"
            f"- Більше шарів ({hl}) → глибша мережа → абстрактніші ознаки, але важче тренувати\n"
            f"- Менший learning rate ({lr}) → повільніше, але стабільніше навчання\n"
            f"- Більший dropout ({dr}) → краще узагальнення, але повільніше тренування\n"
            f"- Batch {bs} і {ep} епох — за стільки кроків модель побачить дані"
        )
        return diagram, text

    inputs = [hidden_layers, neurons, dropout, learning_rate, batch_size, epochs]
    for inp in inputs:
        inp.change(render_architecture, inputs=inputs, outputs=[arch_diagram, params_info])

    return hidden_layers, neurons, dropout, learning_rate, batch_size, epochs, arch_diagram, params_info, render_architecture


def build_tab_training(arch_inputs):
    """Таб 4: одна велика кнопка «Тренувати» + прогрес-бар."""
    hidden_layers, neurons, dropout, learning_rate, batch_size, epochs = arch_inputs
    gr.Markdown(
        "### 🎯 Тренуємо мережу\n"
        "Натисніть кнопку — і мережа почне вчитись з параметрами, які ви налаштували на попередній вкладці. "
        "Це може зайняти від 30 секунд до кількох хвилин."
    )
    train_btn = gr.Button("▶️ Тренувати мережу", variant="primary", size="lg")
    result = gr.Markdown()
    train_btn.click(
        train_model,
        inputs=[hidden_layers, neurons, dropout, learning_rate, batch_size, epochs],
        outputs=result,
    )


def build_tab_epochs():
    """Таб 5: повзунок епохи, передбачення на 10 картинках, ваги першого шару."""
    gr.Markdown(
        "### 🔍 Подивимось, що мережа вивчила в кожній епосі\n"
        "Перетягніть повзунок — і ви побачите, наскільки добре мережа працювала після кожної епохи навчання."
    )
    epoch_slider = gr.Slider(1, 20, value=1, step=1, label="Епоха")
    metrics = gr.Markdown()
    examples = gr.Image(label="10 тестових картинок та передбачення мережі", height=380)
    refresh = gr.Button("🔄 Показати інші 10 картинок")
    weights_img = gr.Image(label="Ваги нейронів першого шару", height=320)
    gr.Markdown(
        "**Що видно на цих картинках?** Кожен квадратик — це один нейрон першого шару. "
        "Ми малюємо його ваги як зображення 28×28 — як ніби сам нейрон «дивиться» на картинку. "
        "🟥 **Червоне** — піксель збільшує активацію нейрона. "
        "🟦 **Синє** — піксель зменшує активацію. "
        "Часто видно, що нейрони реагують на конкретні форми: рукав, підошву, кишеню."
    )

    def show_epoch(ep, _trigger=None):
        if not state.history:
            return "⚠️ Спочатку натренуйте мережу на вкладці «Тренування».", None, None
        ep = min(int(ep), len(state.history))
        h = state.history[ep - 1]
        text = (
            f"### Епоха {h['epoch']} з {len(state.history)}\n"
            f"- **Точність на тренуванні:** {h['train_acc']*100:.2f}%\n"
            f"- **Точність на тесті:** {h['test_acc']*100:.2f}%\n"
            f"- **Втрата на тренуванні:** {h['train_loss']:.4f}\n"
            f"- **Втрата на тесті:** {h['test_loss']:.4f}"
        )
        if state.last_test_indices is None or _trigger == "refresh":
            state.last_test_indices = np.random.choice(len(state.X_test), 10, replace=False)
        idxs = state.last_test_indices
        grid = render_image_grid(
            state.X_test[idxs], state.y_test[idxs],
            h["test_preds"][idxs], h["test_confs"][idxs], ncols=5,
        )
        wimg = render_weight_filters(h["first_layer_weights"], n_show=10)
        return text, grid, wimg

    epoch_slider.change(show_epoch, inputs=epoch_slider, outputs=[metrics, examples, weights_img])
    refresh.click(lambda ep: show_epoch(ep, _trigger="refresh"),
                  inputs=epoch_slider, outputs=[metrics, examples, weights_img])
    return epoch_slider


def build_tab_comparison():
    """Таб 6: графіки точності/втрати, матриця помилок, точність по класах."""
    gr.Markdown(
        "### 📈 Загальна картина\n"
        "Подивимось на тренування цілком: де мережа покращувалась, які класи їй важко й чи "
        "не сталося перенавчання."
    )
    refresh_btn = gr.Button("🔄 Оновити графіки", variant="primary")
    with gr.Row():
        acc_plot = gr.Plot(label="Точність по епохах")
        loss_plot = gr.Plot(label="Втрата по епохах")
    confusion = gr.Plot(label="Матриця помилок")
    per_class = gr.Dataframe(
        headers=["Клас", "Точність, %", "Кількість тестових"],
        label="Точність по класах",
    )
    summary = gr.Markdown()

    def refresh():
        if not state.history:
            return None, None, None, None, "⚠️ Спочатку натренуйте мережу на вкладці «Тренування»."
        acc_fig = render_curve(state.history, "train_acc", "test_acc",
                               "Точність по епохах", "Точність")
        loss_fig = render_curve(state.history, "train_loss", "test_loss",
                                "Втрата по епохах", "Втрата")
        last = state.history[-1]
        cm_fig = render_confusion_matrix(state.y_test, last["test_preds"])
        df = per_class_accuracy_table(state.y_test, last["test_preds"])
        best = max(state.history, key=lambda h: h["test_acc"])
        worst = min(state.history, key=lambda h: h["test_acc"])
        worst_class = df.iloc[-1]
        best_class = df.iloc[0]
        text = (
            f"### Підсумок\n"
            f"- **Найкраща епоха:** {best['epoch']} — точність {best['test_acc']*100:.2f}%\n"
            f"- **Найгірша епоха:** {worst['epoch']} — точність {worst['test_acc']*100:.2f}%\n"
            f"- **Найкраще розпізнається:** {best_class['Клас']} ({best_class['Точність, %']}%)\n"
            f"- **Найгірше розпізнається:** {worst_class['Клас']} ({worst_class['Точність, %']}%)\n\n"
            f"💡 Якщо точність на тренуванні набагато вища за точність на тесті — це **перенавчання**. "
            f"Спробуйте зменшити кількість нейронів або додати dropout."
        )
        return acc_fig, loss_fig, cm_fig, df, text

    refresh_btn.click(refresh, outputs=[acc_plot, loss_plot, confusion, per_class, summary])


def build_tab_summary():
    """Таб 7: підсумок уроку, приклади застосувань, що вивчати далі, результат учня."""
    gr.Markdown(
        "### 🎓 Що ми сьогодні зробили\n\n"
        "**Нейронна мережа** — це програма, яка на прикладах вчиться розпізнавати закономірності.\n\n"
        "Сьогодні ми навчили мережу розпізнавати 10 видів одягу за зображенням 28×28 пікселів. "
        "Мережа дивилася на тисячі прикладів, помилялася, виправляла свої ваги — "
        "і поступово стала вгадувати правильний клас."
    )
    gr.Markdown(
        "### 🌍 Де нейронні мережі використовують у житті\n"
        "- 📱 **Камера телефону** — впізнає обличчя, тварин, текст\n"
        "- 🚗 **Безпілотні авто** — розпізнають пішоходів, знаки, смуги\n"
        "- 🏥 **Медицина** — знаходять пухлини на рентген-знімках\n"
        "- 🌐 **Перекладачі** — Google Translate, DeepL\n"
        "- 🎵 **Музика і відео** — рекомендації Spotify, YouTube, TikTok\n"
        "- 💬 **Чат-боти** — як ChatGPT, Claude\n"
        "- ✈️ **Прогноз погоди й оптимізація логістики**"
    )
    gr.Markdown(
        "### 📚 Що вивчати далі\n"
        "- **PyTorch** — фреймворк, який ми сьогодні використали (pytorch.org/tutorials)\n"
        "- **Kaggle** — змагання й набори даних (kaggle.com)\n"
        "- **fast.ai** — безкоштовний курс «Practical Deep Learning» (course.fast.ai)\n"
        "- **3Blue1Brown** — серія відео про нейронні мережі на YouTube"
    )
    gr.Markdown("### 🏆 Ваш результат")
    show_btn = gr.Button("Показати мої параметри та точність", variant="primary")
    result_table = gr.Dataframe(headers=["Параметр", "Значення"], label="Параметри уроку")
    accuracy_text = gr.Markdown()

    def show_result():
        if not state.history or state.architecture is None:
            return None, "⚠️ Спочатку натренуйте мережу."
        a = state.architecture
        final = state.history[-1]
        rows = [
            ["Прихованих шарів", a["hidden_layers"]],
            ["Нейронів у шарі", a["neurons_per_layer"]],
            ["Dropout", a["dropout"]],
            ["Learning rate", a["learning_rate"]],
            ["Batch size", a["batch_size"]],
            ["Епох", a["num_epochs"]],
        ]
        df = pd.DataFrame(rows, columns=["Параметр", "Значення"])
        text = (
            f"## 🎯 Фінальна точність на тестових даних: **{final['test_acc']*100:.2f}%**\n\n"
            f"Це означає, що з 10 нових картинок ваша мережа правильно вгадає приблизно "
            f"**{round(final['test_acc']*10)}**. Молодці!"
        )
        return df, text

    show_btn.click(show_result, outputs=[result_table, accuracy_text])

## Збираємо додаток із 7 вкладок

In [ ]:
def build_app():
    """Збирає всі вкладки в один Gradio-додаток із збереженням стану між табами."""
    with gr.Blocks(title="Урок з нейронних мереж", theme=gr.themes.Soft()) as demo:
        gr.Markdown(
            "# 🧠 Урок з нейронних мереж — Fashion MNIST\n"
            "Сьогодні ми разом побудуємо й натренуємо нейронну мережу, яка вчиться "
            "розпізнавати одяг. Ідіть по вкладках зліва направо."
        )

        with gr.Tab("1. Що це таке"):
            build_tab_intro()

        with gr.Tab("2. Дані"):
            sample_grid, pixel_view, histogram, status, refresh_data = build_tab_data()
            # При завантаженні табу одразу показуємо приклади.
            demo.load(refresh_data, outputs=[sample_grid, pixel_view, histogram, status])

        with gr.Tab("3. Архітектура"):
            arch = build_tab_architecture()
            hidden_layers, neurons, dropout, learning_rate, batch_size, epochs = arch[:6]
            arch_diagram, params_info, render_architecture = arch[6], arch[7], arch[8]
            # Показуємо схему за стандартними значеннями при відкритті.
            demo.load(
                render_architecture,
                inputs=[hidden_layers, neurons, dropout, learning_rate, batch_size, epochs],
                outputs=[arch_diagram, params_info],
            )

        with gr.Tab("4. Тренування"):
            build_tab_training((hidden_layers, neurons, dropout, learning_rate, batch_size, epochs))

        with gr.Tab("5. Епохи"):
            build_tab_epochs()

        with gr.Tab("6. Порівняння"):
            build_tab_comparison()

        with gr.Tab("7. Підсумок"):
            build_tab_summary()

    return demo


demo = build_app()

## Запуск

Натисніть «Run» нижче — відкриється Gradio-інтерфейс. На Kaggle додайте `share=True`,
щоб отримати публічне посилання.

In [ ]:
demo.launch()